# Managing Metadata with MetaInformationTable

# Introduction

This tutorial demonstrates how to use ``BaseMetaInformationTableSchema`` and ``MetaInformationTableManifestation`` to create a single-row table dedicated to storing application metadata, such as versioning, authors, or configuration states that benefit from caching.

## What is a MetaInformationTable?

A **MetaInformationTable** is a specialized Singleton Table. Like a Singleton Table, it holds exactly one row. However, it adds specific functionality for metadata:
*   **Caching**: The ``MetaInformationTableManifestation`` caches the metadata in memory, reducing database hits for frequently accessed static data.
*   **Simplified Access**: It provides a ``meta_information`` property to access this cached data easily.

This tutorial will guide you through:
- Defining a Metadata Schema Mixin
- Creating a Manifestation with caching
- Setting up the Database
- Initializing and Accessing Metadata
- Updating Metadata (and automatically invalidating the cache)

**Prerequisites:**
- Basic familiarity with Python and SQLAlchemy
- Installed package: ``sqlalchemyobjects``

## Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Usage](#Usage)
- [Async Usage](#Async-Usage)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)



# Importing the Module

We start by importing the necessary classes.



In [ ]:
from pathlib import Path
from sqlalchemy.orm import Mapped, mapped_column, DeclarativeBase
from sqlalchemy.ext.asyncio import AsyncAttrs

from sqlalchemyobjects import Database, BaseMetaInformationTableSchema, MetaInformationTableManifestation

# Core Functionality

## 1. Define the Schema Mixin

We inherit from ``BaseMetaInformationTableSchema``. Unlike ``BaseSingletonTableSchema``, we typically define specific columns for our metadata.



In [ ]:
class AppMetaSchema(BaseMetaInformationTableSchema):
    """Schema for application metadata."""
    version: Mapped[str] = mapped_column(default="0.0.1")
    author: Mapped[str] = mapped_column(default="Unknown")
    last_migration: Mapped[str] = mapped_column(nullable=True)

## 2. Define the Manifestation

The manifestation provides the caching logic. We can optionally provide ``init_info`` during construction or setup.



In [ ]:
class AppMetaManifestation(MetaInformationTableManifestation):
    """Manifestation for AppMeta table, providing cached access to metadata."""


## 3. Define the Database Schema

Combine the TableSchema with the Declarative Base.



In [ ]:
class DatabaseSchema(AsyncAttrs, DeclarativeBase):
    """Declarative base class for the database schema."""

class AppMetaTable(AppMetaSchema, DatabaseSchema):
    """SQLAlchemy table definition for Application Metadata."""
    __tablename__ = "app_metadata"



## 4. Define the Database Class

Register the table in the ``table_map``.



In [ ]:
class MetaDatabase(Database):
    """Database class managing the metadata table."""
    schema = DatabaseSchema
    table_map = {
        "meta": (AppMetaManifestation, AppMetaTable, {})
    }

    @property
    def meta(self) -> AppMetaManifestation:
        return self.tables["meta"]

## 5. Setup Database

Initialize the database. We can pass ``init_info`` to the manifestation during database initialization if we wanted, or set it later.



In [ ]:
db_path = Path("tutorial_meta.sqlite")
if db_path.exists():
    db_path.unlink()

database = MetaDatabase(path=db_path)
database.create_database()

# Access the table
meta_table = database.meta

# Usage

## Initialization

For metadata, we often want to ensure a row exists with default values. The ``build()`` method or ``create_meta_information()`` can be used.



In [ ]:
# Initialize with defaults if not exists
# We can pass specific values to start with
meta_table.create_meta_information(item={"version": "1.0.0", "author": "Tutorial Bot"})

## Reading Metadata (Cached)

The ``meta_information`` property provides read access. It fetches from the DB once and caches the result.



In [ ]:
print("Reading metadata...")
info = meta_table.meta_information
print(f"Version: {info['version']}")
print(f"Author: {info['author']}")

# Subsequent access uses cache (no DB query)
print(f"Version (cached): {meta_table.meta_information['version']}")

## Updating Metadata

When you update metadata using ``set_meta_information``, the cache is automatically cleared/updated.



In [ ]:
print("Updating version...")
meta_table.set_meta_information(item={"version": "1.1.0", "last_migration": "2023-10-27"})

# The next access refetches from DB
new_info = meta_table.meta_information
print(f"New Version: {new_info['version']}")
print(f"Last Migration: {new_info['last_migration']}")

# Async Usage

``MetaInformationTableManifestation`` fully supports async operations.



In [ ]:
import anyio

async def async_meta_demo():
    """Demonstration of using the metadata table in asynchronous mode."""
    async_path = anyio.Path("tutorial_meta_async.sqlite")
    if await async_path.exists():
        await async_path.unlink()

    async_db = MetaDatabase(path=str(async_path), async_engine=True)
    await async_db.create_database_async()

    meta_async = async_db.meta

    # Create Async
    await meta_async.create_meta_information_async(item={"version": "2.0.0"})

    # Get Async (explicit method needed for async retrieval)
    # Note: The .meta_information property is synchronous and relies on a loaded cache.
    # For async, use get_meta_information_async()
    info = await meta_async.get_meta_information_async()
    print(f"Async Version: {info['version']}")

    # Update Async
    await meta_async.set_meta_information_async(item={"version": "2.1.0"})
    info = await meta_async.get_meta_information_async()
    print(f"Async Updated Version: {info['version']}")

    await async_db.close_async()
    await async_path.unlink()

# Run async demo
await async_meta_demo()

In [ ]:
# Cleanup
database.close()
if db_path.exists():
    db_path.unlink()

# API Highlights

- **``BaseMetaInformationTableSchema``**: Inherits from ``BaseSingletonTableSchema``. Define your metadata columns here.
- **``MetaInformationTableManifestation``**:
    - **``meta_information``**: Property that returns the cached dictionary of metadata.
    - **``create_meta_information(item)``**: Creates the row if missing.
    - **``set_meta_information(item)``**: Updates the row and invalidates cache.



# Troubleshooting / FAQs

- **Problem**: `KeyError` when accessing `meta_table.meta_information['some_key']`.
  - **Solution**: Ensure the column exists in your Schema and that the row has been created. If the column is nullable and None, the key will exist but value will be None.



# Conclusion and Next Steps

You've learned how to use ``MetaInformationTable`` to manage application metadata efficiently with caching.

- **Reference**: See ``docs/concepts/comprehensive.rst``.

